# Pyrates FEP
# Notebook 02: Game assignation to the experts

### Quick reference: what to tune

| Parameter | Where | Default | What it does |
|---|---|---|---|
| `EXPERT_CAPS` | section 3 (`gen-10`) | 8 experts | **Add/remove an expert here.** A `{name: cap}` dict - `cap` is that expert's max total feedback instances annotated. Nothing else in the notebook needs to change. |
| `MIN_FEEDBACK_PER_GAME` | section 2 | `5` | A game needs at least this many feedback instances to be eligible for assignment. |
| `MAX_FEEDBACK_PER_GAME` | section 2 | `33` | Optional upper bound on feedback instances per game; `None` = no cap. |
| `MAX_GAMES_PER_PAIR` | section 3 (`gen-10`) | `10` | Ceiling on how many times the same duo of experts can be paired. Only a ceiling - the pair-selection logic already prefers fresh pairs first, so reuse happens only when needed. |
| `ASSIGNMENT_SEED` | section 3 (`gen-10`) | `26` | Single seed for every random choice in section 3 (bundle starting order, pair tie-breaks). Change it to get a different (still valid) assignment; keep it fixed for reproducible results. |
| `BALANCE_LEVEL_DISTRIBUTION` | section 3 (`gen-11`) | `True` | If `True`, bundle order also keeps the selected feedback's `level_id` mix close to the `level_id` mix of the whole eligible pool (see the level-balance check in section 4). |
| `BC_BALANCE_WEIGHT` | section 3 (`gen-11`) | `8.0` | How strongly bundle order favors keeping the running `B` vs `C` total even (see section 4's B/C balance checks). Tuned empirically - rarely needs changing. |

Re-run the whole notebook top to bottom after changing any of these - later cells depend on earlier
ones. Section 4 ("Checks") reports the effect of your changes: workload per expert, B/C balance,
pair variety, and level_id balance.

### Inputs: 
1) Experts
- See Section 3
  

2) Interaction traces
- `data/cleaned/cleaned_interaction_traces.csv`

### Definitions and scope

- **Feedback instance**: one `RECEIVED` trace under `object_name == "ASSISTANT_HELP"`, i.e. one
  piece of LLM feedback actually delivered to a student (an unanswered `ASKED` has nothing an
  expert could rate, so it is not counted).
- **Eligible games**: games in group `B` or `C` (the agent is absent in `A`, there is nothing to
  annotate there) whose number of feedback instances (`n_received`) falls within
  `[MIN_FEEDBACK_PER_GAME, MAX_FEEDBACK_PER_GAME]` (see section 2 - `MAX_FEEDBACK_PER_GAME` can be
  left as `None` for no upper bound).
- **Assignment unit**: a whole `game_id`. An expert who is assigned a game annotates every
  feedback instance of that session, never a subset of it.

### Constraints

| Constraint | Rule |
|---|---|
| Workload cap | each expert has their own cap on total feedback instances annotated (see `EXPERT_CAPS` below - add a new expert by adding one entry there) |
| Double annotation | every assigned game is annotated by exactly **2** experts |
| Per-expert balance | for a given expert, the number of `B` instances and the number of `C` instances they annotate are as close as possible |
| Overall balance | summed across all experts, the total `B` instances annotated and the total `C` instances annotated are as close as possible |
| Varied pairing | the same duo of experts is not reused: for `n` experts there are `n * (n - 1) / 2` possible pairs, and a pair is used **at most once** |

Not every eligible game needs to be assigned: only enough games to fill the experts' workload
under these constraints.

## 1. Setup and imports

In [264]:
from __future__ import annotations

import itertools
import random
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths relative to the notebook: notebooks/ -> project root -> data/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
INPUT_PATH = DATA_DIR / "cleaned" / "cleaned_interaction_traces.csv"
OUTPUT_PATH = DATA_DIR / "cleaned" / "expert_assignment.csv"

print("Project root :", PROJECT_ROOT.name)
print("Input        :", INPUT_PATH.relative_to(PROJECT_ROOT), "| exists:", INPUT_PATH.exists())
print("Output       :", OUTPUT_PATH.relative_to(PROJECT_ROOT))

Project root : Pyrates_FEP
Input        : data\cleaned\cleaned_interaction_traces.csv | exists: True
Output       : data\cleaned\expert_assignment.csv


## 2. Eligible games

In [265]:
MIN_FEEDBACK_PER_GAME = 5     # a game needs at least this many feedback instances to be eligible
MAX_FEEDBACK_PER_GAME = 30  # optional upper bound on feedback instances per game (None = no cap)

traces = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, low_memory=False)

games = traces.drop_duplicates("game_id")[["game_id", "group_id"]]

feedback = traces[traces["object_name"] == "ASSISTANT_HELP"]
n_received = (
    feedback[feedback["action_name"] == "RECEIVED"]
    .groupby("game_id")
    .size()
    .rename("n_received")
)

games = games.merge(n_received, on="game_id", how="left")
games["n_received"] = games["n_received"].fillna(0).astype(int)

in_group = games["group_id"].isin(["B", "C"])
above_min = games["n_received"] >= MIN_FEEDBACK_PER_GAME
below_max = games["n_received"] <= MAX_FEEDBACK_PER_GAME if MAX_FEEDBACK_PER_GAME is not None else True

eligible = games[in_group & above_min & below_max].reset_index(drop=True)
excluded_by_max = int((in_group & above_min & ~below_max).sum()) if MAX_FEEDBACK_PER_GAME is not None else 0

print(f"Games total          : {len(games)}")
print(f"Games in B/C         : {int(in_group.sum())}")
print(f"Acceptable interval  : [{MIN_FEEDBACK_PER_GAME}, {MAX_FEEDBACK_PER_GAME if MAX_FEEDBACK_PER_GAME is not None else '∞'}] feedback instances per game")
if MAX_FEEDBACK_PER_GAME is not None:
    print(f"Games excluded (> max): {excluded_by_max}")
print(f"Eligible games       : {len(eligible)} "
      f"(B={int((eligible.group_id == 'B').sum())}, C={int((eligible.group_id == 'C').sum())})")
print(f"Feedback instances available: {int(eligible['n_received'].sum())} "
      f"(B={int(eligible.loc[eligible.group_id == 'B', 'n_received'].sum())}, "
      f"C={int(eligible.loc[eligible.group_id == 'C', 'n_received'].sum())})")

Games total          : 293
Games in B/C         : 193
Acceptable interval  : [5, 30] feedback instances per game
Games excluded (> max): 18
Eligible games       : 122 (B=73, C=49)
Feedback instances available: 1704 (B=1061, C=643)


In [266]:
eligible.groupby("group_id")["n_received"].describe()

,count,mean,std,min,25%,50%,75%,max
group_id,,,,,,,,
B,73.0,14.534247,6.902258,5.0,8.0,14.0,20.0,30.0
C,49.0,13.122449,6.505231,5.0,7.0,12.0,17.0,28.0


## 3. Assignment: matched bundles handed out round-robin

A simple heuristic, no solver involved.

**Step 1 - bundle B with C games of comparable size.** Group `B` has more eligible games than
group `C` (matching them by raw rank would only ever pair `C` against `B`'s largest games and bias
every bundle towards `B`), so games are matched by **percentile rank** within their own group
instead: the largest `C` game pairs with the largest `B` game, the median `C` game with the median
`B` game, and so on. Each `(b_game, c_game)` **bundle** therefore carries roughly the same number
of `B` and `C` instances - but that only keeps *each individual bundle* balanced; it says nothing
about which *subset* of bundles ends up selected once workload caps kick in, and that subset can
still drift towards more `B` than `C` overall (or vice versa). So bundles are then greedily
reordered to keep the *running* `B` vs `C` total close to even as they get placed - not just
size-sorted, since that would always prioritise whichever games happen to be biggest under the
current `MIN`/`MAX_FEEDBACK_PER_GAME` bounds (e.g. games sitting right at `MAX_FEEDBACK_PER_GAME`
would always get picked first, no matter what that bound is set to). If `BALANCE_LEVEL_DISTRIBUTION`
is turned on, that same greedy reorder also keeps the `level_id` mix of everything selected so far
close to the `level_id` mix of the whole eligible pool - tested, this doesn't come at the expense of
the `B`/`C` balance (see the checks in section 4).

**Step 2 - hand out bundles round-robin, in three passes.** Both games of a bundle always go to the
*same* pair of experts, so a pair's contribution to each of its two experts is already close to
balanced by construction. **Phase 1** makes one attempt per bundle, in the order from step 1.
**Phase 2** revisits whatever bundles phase 1 skipped, smallest-first, since a bundle skipped only
because it didn't line up with someone's remaining capacity at the time might still fit once
capacity elsewhere is used up. **Phase 3** mops up any capacity too small for even the smallest
bundle (bundles are at least `2 x MIN_FEEDBACK_PER_GAME`) with single, unmatched games - this can
only push a pair's B/C balance further apart, so it actively prefers whichever pair is currently
short on that particular group. Together, phases 2-3 get the total covered feedback closer to the
theoretical ceiling of `sum(EXPERT_CAPS) // 2` (each bundle spends its size out of *two* experts'
budgets at once, but only counts once towards "feedback instances covered") - some of that ceiling
can still be structurally unreachable (e.g. a leftover of 2 or 3 can never fit a game once
`MIN_FEEDBACK_PER_GAME` is 5).

In every phase, which valid pair - both experts still under their own cap in `EXPERT_CAPS`, and not
already used together more than `MAX_GAMES_PER_PAIR` times - gets a bundle is decided, in order, by:
**(1)** the pair used the fewest times so far - usually not at all yet. This has to come before
load-balance, not after: reusing whichever of two already-active pairs currently has the smaller
total is nearly always cheaper than trying a fresh mix (combining two differently-loaded experts'
totals is rarely lower than doubling the cheaper pair's own total), so if load-balance were the
deciding factor, the same couple of pairs would keep winning and `MAX_GAMES_PER_PAIR` would just be
a ceiling nothing ever pushes against; **(2)** among equally-fresh pairs, the lowest combined
workload, so every expert stays in rotation; **(3)** the pair with the most combined cap - low-cap
experts shouldn't have a single huge bundle swallow most of their budget with one partner, leaving
them with only that one partner's games; **(4)** a random pick (seeded by `ASSIGNMENT_SEED`, so
the result stays reproducible). A handful of bundles (a very large `B` game matched with a very
large `C` game) exceed even the largest cap and are simply left unassigned.

In [ ]:
MAX_GAMES_PER_PAIR = 10    # a given duo of experts is used at most this many times (see step 2 -
                            # this is only a ceiling; step 2 also actively prefers fresh pairs, so
                            # reuse happens only when it helps balance the workload)
ASSIGNMENT_SEED = 2026       # single seed for every random choice in section 3 (bundle order, pair
                            # tie-breaks) - keeps results reproducible; change it to get a different
                            # (still valid) assignment

# Each expert's own workload cap (max feedback instances they annotate in total). Add a new
# expert by adding one entry here - nothing else in the notebook needs to change.
EXPERT_CAPS = {
    "Michael": 120,
    "Magic": 120,
    "Larry": 120,
    "Charles": 120,
    "Scottie": 90,
    "Patrick": 90,
    "David": 60,
    "Clyde": 60,
}

EXPERTS = list(EXPERT_CAPS)
PAIRS = list(itertools.combinations(EXPERTS, 2))

print(f"Experts: {len(EXPERTS)} | possible pairs: {len(PAIRS)}")
for expert in EXPERTS:
    print(f"  {expert}: cap={EXPERT_CAPS[expert]}")

Experts: 8 | possible pairs: 28
  Marine: cap=120
  Mai: cap=120
  Rev: cap=120
  Matthieu: cap=120
  SebJ: cap=90
  Amel: cap=90
  Guil: cap=60
  SebL: cap=60


In [ ]:
BALANCE_LEVEL_DISTRIBUTION = True  # if True, bundle order also keeps the level_id mix of the
                                    # feedback selected so far close to the level_id mix of the
                                    # whole eligible pool (approximately - still greedy, not exact)

size_of = dict(zip(eligible["game_id"], eligible["n_received"]))
b_games = eligible[eligible.group_id == "B"].sort_values("n_received", ascending=False)["game_id"].tolist()
c_games = eligible[eligible.group_id == "C"].sort_values("n_received", ascending=False)["game_id"].tolist()
n_b, n_c = len(b_games), len(c_games)

used_b_index: set[int] = set()
bundles: list[tuple[str, str]] = []
for c_rank, c_id in enumerate(c_games):
    # nearest still-free B game to this C game's percentile rank in the (longer) B list
    target = round(c_rank * (n_b - 1) / (n_c - 1)) if n_c > 1 else 0
    b_index = target
    while b_index in used_b_index and b_index < n_b - 1:
        b_index += 1
    while b_index in used_b_index and b_index >= 0:
        b_index -= 1
    if b_index < 0 or b_index in used_b_index:
        continue
    used_b_index.add(b_index)
    bundles.append((b_games[b_index], c_id))

# starting order is random, not size-sorted (sorting largest-first would always prioritise
# whichever games happen to be biggest under the current MIN/MAX_FEEDBACK_PER_GAME bounds); this
# is really just the seed for the greedy reorder below, which dominates the final order
bundle_rng = random.Random(ASSIGNMENT_SEED)
bundle_rng.shuffle(bundles)

# per-game level_id counts among eligible games' feedback, and the resulting level_id mix across
# the whole eligible pool - the target that BALANCE_LEVEL_DISTRIBUTION tries to track
received = feedback[(feedback["action_name"] == "RECEIVED") & feedback["game_id"].isin(eligible["game_id"])].copy()
received["level_id"] = received["level_id"].astype(int)
level_counts_of: dict[str, Counter] = {
    game_id: Counter(rows["level_id"]) for game_id, rows in received.groupby("game_id")
}
target_level_counts = Counter()
for counts in level_counts_of.values():
    target_level_counts.update(counts)
target_level_total = sum(target_level_counts.values())
target_level_frac = {lvl: cnt / target_level_total for lvl, cnt in target_level_counts.items()}

def bundle_level_counts(bc):
    b_id, c_id = bc
    return level_counts_of.get(b_id, Counter()) + level_counts_of.get(c_id, Counter())

def level_l1_distance(counts):
    total = sum(counts.values())
    if total == 0:
        return sum(target_level_frac.values())
    levels = set(target_level_frac) | set(counts)
    return sum(abs(counts.get(lvl, 0) / total - target_level_frac.get(lvl, 0)) for lvl in levels)

# BC_BALANCE_WEIGHT: percentile-rank matching (step 1 above) keeps each *individual* bundle's B
# and C sizes close, but says nothing about which *subset* of bundles ends up selected once
# workload caps kick in - that subset can still drift towards more B than C (or vice versa).
# Weighted at 8, tested against several random starting orders, this reliably keeps the running
# B/C total close (imbalance ~6 out of ~380 total feedback covered, down from ~36 without it) at
# no cost to level_id balance - the two objectives aren't in tension here. The weight matters
# because the B/C term is a 0-1 fraction while the level_id term maxes out around 2; without
# weighting it up, a tiny level_id improvement could win over a real B/C gap.
BC_BALANCE_WEIGHT = 8.0

# greedy: repeatedly pick, among bundles not yet placed, whichever keeps the *cumulative* B/C
# balance (of everything placed so far, this bundle included) closest to even, and - when
# BALANCE_LEVEL_DISTRIBUTION is on - also keeps the cumulative level_id mix closest to the target
remaining = list(bundles)
ordered: list[tuple[str, str]] = []
cumulative: Counter = Counter()
cum_b, cum_c = 0, 0
for _ in range(len(bundles)):
    best_bc, best_key = None, None
    for bc in remaining:
        b_id, c_id = bc
        new_b, new_c = cum_b + size_of[b_id], cum_c + size_of[c_id]
        bc_gap = abs(new_b - new_c) / (new_b + new_c) if (new_b + new_c) else 0
        level_term = level_l1_distance(cumulative + bundle_level_counts(bc)) if BALANCE_LEVEL_DISTRIBUTION else 0
        key = BC_BALANCE_WEIGHT * bc_gap + level_term
        if best_key is None or key < best_key:
            best_key, best_bc = key, bc
    ordered.append(best_bc)
    b_id, c_id = best_bc
    cumulative += bundle_level_counts(best_bc)
    cum_b += size_of[b_id]
    cum_c += size_of[c_id]
    remaining.remove(best_bc)
bundles = ordered

bundle_sizes = pd.Series([size_of[b] + size_of[c] for b, c in bundles])
rounded_target_frac = {k: round(v, 3) for k, v in sorted(target_level_frac.items())}
print(f"Bundles formed: {len(bundles)} (limited by the smaller group, C, with {n_c} games)")
print(f"Bundle order        : B/C-balance greedy{' + level-balance' if BALANCE_LEVEL_DISTRIBUTION else ''}")
print(f"Target level_id mix (eligible pool): {rounded_target_frac}")
print("Bundle size (B instances + C instances):")
print(bundle_sizes.describe().to_string())

In [269]:
pair_rng = random.Random(ASSIGNMENT_SEED)

load = {expert: {"B": 0, "C": 0} for expert in EXPERTS}
pair_used = {pair: 0 for pair in PAIRS}
assigned_rows = []
used_bundles: set[tuple[str, str]] = set()
assigned_game_ids: set[str] = set()

def combined_load(pair):
    e1, e2 = pair
    return load[e1]["B"] + load[e1]["C"] + load[e2]["B"] + load[e2]["C"]

def combined_cap(pair):
    e1, e2 = pair
    return EXPERT_CAPS[e1] + EXPERT_CAPS[e2]

def pick_pair(valid_pairs):
    # Ties are broken, in order, by:
    #  1. preferring a pair that has been used the fewest times so far (usually not at all yet).
    #     This has to come *before* load-balance: reusing whichever of two already-active pairs
    #     currently has the smaller total is nearly always cheaper (in combined_load terms) than
    #     trying a brand-new mix, because combining two differently-loaded experts' totals is
    #     rarely lower than doubling the cheaper pair's own total. So if load-balance were the
    #     primary key, the same couple of pairs would keep winning and MAX_GAMES_PER_PAIR would
    #     just be a ceiling nothing ever pushed against - variety has to be decided first.
    #  2. least combined workload, among equally-fresh pairs, so every expert stays in rotation
    #  3. preferring the pair with the most combined cap - low-cap experts shouldn't have a
    #     single huge bundle swallow most of their budget with one partner, leaving them with
    #     only that one partner's games; higher-cap pairs can better absorb a big bundle and
    #     still have room left for more.
    #  4. random (seeded by ASSIGNMENT_SEED, so the result stays reproducible)
    min_used = min(pair_used[p] for p in valid_pairs)
    tied_pairs = [p for p in valid_pairs if pair_used[p] == min_used]
    min_load = min(combined_load(p) for p in tied_pairs)
    tied_pairs = [p for p in tied_pairs if combined_load(p) == min_load]
    max_cap = max(combined_cap(p) for p in tied_pairs)
    tied_pairs = [p for p in tied_pairs if combined_cap(p) == max_cap]
    return pair_rng.choice(tied_pairs)

def pick_pair_for_group(valid_pairs, group_id):
    # like pick_pair, but for a single, unmatched game (phase 3 below): since it only feeds one
    # group's total, unlike a matched bundle it can push a pair's B vs C balance further apart
    # instead of closer. So right after the freshness preference, prefer whichever pair is
    # currently most short on this particular group relative to the other - closing an existing
    # B/C gap instead of (by sheer luck of who has spare cap) blindly widening one.
    other_group = "C" if group_id == "B" else "B"
    def deficit(p):
        e1, e2 = p
        return (load[e1][other_group] + load[e2][other_group]) - (load[e1][group_id] + load[e2][group_id])

    min_used = min(pair_used[p] for p in valid_pairs)
    tied_pairs = [p for p in valid_pairs if pair_used[p] == min_used]
    max_deficit = max(deficit(p) for p in tied_pairs)
    tied_pairs = [p for p in tied_pairs if deficit(p) == max_deficit]
    min_load = min(combined_load(p) for p in tied_pairs)
    tied_pairs = [p for p in tied_pairs if combined_load(p) == min_load]
    max_cap = max(combined_cap(p) for p in tied_pairs)
    tied_pairs = [p for p in tied_pairs if combined_cap(p) == max_cap]
    return pair_rng.choice(tied_pairs)

def valid_pairs_needing(size):
    candidates = [e for e in EXPERTS if load[e]["B"] + load[e]["C"] + size <= EXPERT_CAPS[e]]
    return [p for p in itertools.combinations(candidates, 2) if pair_used[p] < MAX_GAMES_PER_PAIR]

def try_assign_bundle(b_id, c_id):
    valid_pairs = valid_pairs_needing(size_of[b_id] + size_of[c_id])
    if not valid_pairs:
        return False  # bundle too large, or every eligible pair already full / already used
    expert_1, expert_2 = pick_pair(valid_pairs)

    for expert in (expert_1, expert_2):
        load[expert]["B"] += size_of[b_id]
        load[expert]["C"] += size_of[c_id]
    pair_used[(expert_1, expert_2)] += 1
    used_bundles.add((b_id, c_id))
    assigned_game_ids.update((b_id, c_id))

    assigned_rows.append({"game_id": b_id, "group_id": "B", "n_received": size_of[b_id], "expert_1": expert_1, "expert_2": expert_2})
    assigned_rows.append({"game_id": c_id, "group_id": "C", "n_received": size_of[c_id], "expert_1": expert_1, "expert_2": expert_2})
    return True

def try_assign_single(game_id, group_id):
    size = size_of[game_id]
    valid_pairs = valid_pairs_needing(size)
    if not valid_pairs:
        return False
    expert_1, expert_2 = pick_pair_for_group(valid_pairs, group_id)

    for expert in (expert_1, expert_2):
        load[expert][group_id] += size
    pair_used[(expert_1, expert_2)] += 1
    assigned_game_ids.add(game_id)

    assigned_rows.append({"game_id": game_id, "group_id": group_id, "n_received": size, "expert_1": expert_1, "expert_2": expert_2})
    return True

# Phase 1: main pass, one attempt per bundle, in the order chosen in section 3 step 1
for b_id, c_id in bundles:
    try_assign_bundle(b_id, c_id)
bundles_covered_main_pass = len(used_bundles)

# Phase 2: top-up over bundles the main pass skipped, smallest-first (best chance of fitting into
# whatever small gaps are left in someone's cap)
leftover_bundles = sorted((bc for bc in bundles if bc not in used_bundles), key=lambda bc: size_of[bc[0]] + size_of[bc[1]])
for b_id, c_id in leftover_bundles:
    try_assign_bundle(b_id, c_id)
bundles_covered_topup = len(used_bundles) - bundles_covered_main_pass

# Phase 3: individual-game top-up. Bundles have a minimum size of 2 x MIN_FEEDBACK_PER_GAME, so a
# lot of leftover per-expert capacity can be smaller than that and still go entirely unused after
# phases 1-2. A single still-eligible game (not matched with one from the other group) can fit
# those smaller gaps - this still respects the notebook's definition that the assignment unit is a
# whole game_id, just without the B+C size-matching. pick_pair_for_group actively steers these
# towards whichever pair is short on that particular group, so this mop-up closes B/C imbalance
# instead of risking widening it.
leftover_games = eligible[~eligible["game_id"].isin(assigned_game_ids)].sort_values("n_received")
singles_assigned = 0
for _, row in leftover_games.iterrows():
    if try_assign_single(row["game_id"], row["group_id"]):
        singles_assigned += 1

assignment = pd.DataFrame(assigned_rows).sort_values("game_id").reset_index(drop=True)
theoretical_max = sum(EXPERT_CAPS.values()) // 2

print(f"Bundles assigned  : {len(used_bundles)} / {len(bundles)} "
      f"({bundles_covered_main_pass} main pass + {bundles_covered_topup} bundle top-up)")
print(f"Single-game top-up: {singles_assigned} games")
print(f"Games assigned          : {len(assignment)} / {len(eligible)}")
print(f"Feedback instances covered: {int(assignment['n_received'].sum())} / {theoretical_max} theoretical max "
      f"(sum of caps / 2) / {int(eligible['n_received'].sum())} available")
assignment.sort_values("n_received", ascending=False)

Bundles assigned  : 18 / 49 (18 main pass + 0 bundle top-up)
Single-game top-up: 2 games
Games assigned          : 38 / 122
Feedback instances covered: 381 / 390 theoretical max (sum of caps / 2) / 1704 available


,game_id,group_id,n_received,expert_1,expert_2
31,p6UCEdQ,B,23,Matthieu,Amel
32,qqyGKF7,C,22,Matthieu,Amel
10,GAnpFJy,B,18,Marine,Mai
9,Cb8yGv3,B,16,SebJ,Amel
28,cMC1A7b,B,16,Rev,SebJ
13,JrdL7n5,C,16,Marine,Mai
24,Z6zpbKS,C,15,SebJ,Amel
17,PsRCVfX,B,15,Marine,Mai
7,BjMKXrN,C,14,Rev,SebJ
20,TW5ty6U,B,13,Rev,Matthieu


## 4. Checks

In [270]:
# Every game assigned to exactly 2 distinct experts, and only once
assert (assignment["expert_1"] != assignment["expert_2"]).all()
assert not assignment["game_id"].duplicated().any()

# No pair used for more than MAX_GAMES_PER_PAIR bundles (each bundle contributes 2 rows: its B
# game and its C game, both to the same pair - so pair_used, tracked per bundle during
# assignment, is the correct count to check, not a recount of individual game rows)
distinct_pairs_used = sum(1 for count in pair_used.values() if count > 0)
assert max(pair_used.values()) <= MAX_GAMES_PER_PAIR

print(f"Distinct pairs used: {distinct_pairs_used} / {len(PAIRS)} possible")
print("Max times any single pair is reused:", max(pair_used.values()))

Distinct pairs used: 19 / 28 possible
Max times any single pair is reused: 2


In [ ]:
per_expert = []
for expert in EXPERTS:
    is_expert = (assignment["expert_1"] == expert) | (assignment["expert_2"] == expert)
    mine = assignment[is_expert]
    b = int(mine.loc[mine.group_id == "B", "n_received"].sum())
    c = int(mine.loc[mine.group_id == "C", "n_received"].sum())
    per_expert.append(
        {"expert": expert, "cap": EXPERT_CAPS[expert], "games": int(is_expert.sum()),
         "B": b, "C": c, "total": b + c, "imbalance": abs(b - c)}
    )

loads = pd.DataFrame(per_expert)
assert (loads["total"] <= loads["cap"]).all(), loads[loads["total"] > loads["cap"]]

print(loads.to_string(index=False))
print(f"\nPer-expert B/C totals summed across all experts (each covered instance is counted twice -"
      f" once per annotator): B={int(loads['B'].sum())} | C={int(loads['C'].sum())} "
      f"| overall imbalance: {abs(int(loads['B'].sum()) - int(loads['C'].sum()))}")
print(f"Max per-expert B/C imbalance: {int(loads['imbalance'].max())}")

# unique feedback instances covered, by group - each game_id appears exactly once in `assignment`
# regardless of how many experts annotate it, so this is the real (non-double-counted) B/C split
unique_b = int(assignment.loc[assignment.group_id == "B", "n_received"].sum())
unique_c = int(assignment.loc[assignment.group_id == "C", "n_received"].sum())
print(f"\nUnique feedback instances covered: B={unique_b} | C={unique_c} | total={unique_b + unique_c} "
      f"| B/C imbalance: {abs(unique_b - unique_c)}")

print("\nOK: every expert stayed within their own cap, and every expert paired at most once with any other expert.")

In [272]:
# Flag any two experts whose assigned game sets fully overlap - if their only shared bundle is
# also their *only* bundle each, they end up annotating exactly the same games, which defeats the
# point of pairing different experts together
games_of = {
    expert: set(assignment.loc[(assignment["expert_1"] == expert) | (assignment["expert_2"] == expert), "game_id"])
    for expert in EXPERTS
}
identical_pairs = [
    (e1, e2) for e1, e2 in PAIRS
    if games_of[e1] and games_of[e1] == games_of[e2]
]
if identical_pairs:
    print("WARNING: these expert pairs annotate the exact same set of games:")
    for e1, e2 in identical_pairs:
        print(f"  {e1} & {e2}: {sorted(games_of[e1])}")
else:
    print("OK: no two experts have fully identical game assignments.")

OK: no two experts have fully identical game assignments.


In [273]:
# How closely the level_id mix of the actually-selected feedback tracks the target (the level_id
# mix across the whole eligible pool) - informative regardless of BALANCE_LEVEL_DISTRIBUTION, so
# you can see whether turning it on actually helps
selected_level_counts = Counter()
for game_id in assignment["game_id"]:
    selected_level_counts.update(level_counts_of.get(game_id, Counter()))
selected_level_total = sum(selected_level_counts.values())

level_balance = pd.DataFrame(
    {
        "target_frac": pd.Series(target_level_frac),
        "selected_frac": pd.Series({lvl: cnt / selected_level_total for lvl, cnt in selected_level_counts.items()}),
    }
).fillna(0).sort_index()
level_balance["abs_diff"] = (level_balance["target_frac"] - level_balance["selected_frac"]).abs()

print(f"BALANCE_LEVEL_DISTRIBUTION = {BALANCE_LEVEL_DISTRIBUTION}")
print(level_balance.round(3).to_string())
print(f"Total absolute deviation (L1, lower is better): {level_balance['abs_diff'].sum():.3f}")

BALANCE_LEVEL_DISTRIBUTION = True
   target_frac  selected_frac  abs_diff
1        0.103          0.118     0.015
2        0.065          0.058     0.007
3        0.254          0.281     0.027
4        0.307          0.297     0.010
5        0.130          0.121     0.009
6        0.072          0.073     0.002
7        0.034          0.029     0.005
8        0.036          0.024     0.012
Total absolute deviation (L1, lower is better): 0.087


## 5. Output

One comma-separated list of `game_id` per expert (an expert's games are those where they appear as
`expert_1` or `expert_2`), plus the full assignment table exported to
`data/cleaned/expert_assignment.csv` for downstream use by the annotation tool.

In [274]:
for expert in EXPERTS:
    is_expert = (assignment["expert_1"] == expert) | (assignment["expert_2"] == expert)
    game_ids = assignment.loc[is_expert, "game_id"].tolist()
    print(f"{expert} : {','.join(game_ids)}")

Marine : GAnpFJy,JrdL7n5,KMHUEfY,PMcjY1u,PsRCVfX,VBJHtCU,ZWNLrMz,b4cAb8f,icGxYBf,sPe8xT6,sWWsfut,wk49Atk
Mai : 1wY5pLR,39n6bvR,9dQ8q87,CDMMmNw,GAnpFJy,HWfSquy,JrdL7n5,PsRCVfX,X2TaH4S,icGxYBf,kirL3gb,sSB7JBV
Rev : 39n6bvR,6stGLrS,9dQ8q87,BjMKXrN,KHmqQwy,KMHUEfY,S8zuqdE,TW5ty6U,W7Rp7uy,cMC1A7b,rzmHDWh,wk49Atk
Matthieu : 6stGLrS,AyVKNE4,PMcjY1u,SPCYvuq,TW5ty6U,ZWNLrMz,byCcupD,kirL3gb,p6UCEdQ,qqyGKF7,sSB7JBV
SebJ : BjMKXrN,Cb8yGv3,J6davsE,VBJHtCU,Z6zpbKS,byCcupD,cMC1A7b,sPe8xT6
Amel : Cb8yGv3,HWfSquy,X2TaH4S,Z6zpbKS,p6UCEdQ,qqyGKF7
Guil : 1Y2RPm6,1wY5pLR,9NeYms9,CDMMmNw,J6davsE,KHmqQwy,S8zuqdE
SebL : 1Y2RPm6,9NeYms9,AyVKNE4,SPCYvuq,W7Rp7uy,b4cAb8f,rzmHDWh,sWWsfut


In [ ]:
print("Feedback instances annotated per expert, by group:")
display(loads[["expert", "cap", "B", "C", "total"]].rename(columns={"B": "B_feedback", "C": "C_feedback", "total": "total_feedback"}))

print(f"\nTotal B feedback annotated, summed across experts (double-counted - each instance has 2 annotators): {int(loads['B'].sum())}")
print(f"Total C feedback annotated, summed across experts (double-counted - each instance has 2 annotators): {int(loads['C'].sum())}")
print(f"\nUnique B feedback instances covered: {unique_b}")
print(f"Unique C feedback instances covered: {unique_c}")

In [276]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
assignment.to_csv(OUTPUT_PATH, index=False, encoding="utf-8", lineterminator="\n")
print(f"Written: {OUTPUT_PATH.relative_to(PROJECT_ROOT)} ({len(assignment)} rows)")

Written: data\cleaned\expert_assignment.csv (38 rows)


## 6. Summary

In [ ]:
summary = pd.DataFrame(
    [
        {"step": "Eligible games (B/C, >=1 feedback instance)", "value": len(eligible)},
        {"step": "Feedback instances available", "value": int(eligible["n_received"].sum())},
        {"step": "Bundles formed (B game + comparable C game)", "value": len(bundles)},
        {"step": "Games assigned", "value": len(assignment)},
        {"step": "Feedback instances covered (unique)", "value": int(assignment["n_received"].sum())},
        {"step": "  - of which B (unique)", "value": unique_b},
        {"step": "  - of which C (unique)", "value": unique_c},
        {"step": "  - unique B/C imbalance", "value": abs(unique_b - unique_c)},
        {"step": "Distinct expert pairs used", "value": distinct_pairs_used},
        {"step": "Total B instance-annotations (double-counted, summed across experts)", "value": int(loads["B"].sum())},
        {"step": "Total C instance-annotations (double-counted, summed across experts)", "value": int(loads["C"].sum())},
        {"step": "Max expert workload", "value": int(loads["total"].max())},
        {"step": "Min expert workload", "value": int(loads["total"].min())},
        {"step": "Max per-expert B/C imbalance", "value": int(loads["imbalance"].max())},
    ]
)
summary

In [278]:
len(feedback[(feedback["game_id"] == "GLLtefW") & (feedback["action_name"] == "RECEIVED")])

3